In [ ]:
import os
import random
import logging
import asyncio
from telegram import Update
from telegram.ext import Application, CommandHandler, MessageHandler, filters, ContextTypes, ConversationHandler
from telegram.constants import ParseMode

# Enable logging
logging.basicConfig(
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s', level=logging.INFO
)
logger = logging.getLogger(__name__)

# Conversation states
QUESTION_1, QUESTION_2, QUESTION_3, QUESTION_4, QUESTION_5, QUESTION_6, QUESTION_7, QUESTION_8, QUESTION_9, QUESTION_10 = range(10)

# Incorrect responses collection
INCORRECT_RESPONSES = [
    "Интересно, чем сейчас занимается гэгэ\\.",
    "А я бы мог это время потратить на новую статую Его Величества\\.",
    "Нужно купить подарки принцу\\.",
    "Это Вы так хорошо знаете Айко? Как интересно, хах\\.",
    "Даже никчемные помощники принца бы знали, что это неправильный ответ\\.",
    "Не совсем, попробуйте ещё\\.",
    "Звучит как-то странно\\.",
    "*\\*зевок\\** Стоит купить мантоу для гэгэ\\."
]


class DiceFilter(filters.MessageFilter):
    def filter(self, message):
        return bool(message.dice)

DICE_FILTER = DiceFilter()


# Start command
async def start(update: Update, context: ContextTypes.DEFAULT_TYPE) -> int:
    """Send message on `/start`."""
    user = update.message.from_user
    logger.info("User %s started the conversation.", user.first_name)

    # Send the first question as a poem
    text1 = (
        "Добро пожаловать в Призрачный Город\. По просьбе Айко я буду помогать вам с вашим квестом\. Вам нужно будет решить пару загадок, и ответить на них правильно, ничего сложного, даже шиди моего подопечного справился бы с этим спустя рукава\.\n"
        "\n"
        "\n"
        "\n"
    )
    text2 = (
        "Ваша первая загадка такова: \n"
        "\n"
    )
    poem = (
        "\"_Дар небес, но не из плоти, не из крови_\n"
        "_Лишь бросок вершит судьбы мосты\\._\n"
        "_Слышен шум ночной призрачного града_\n"
        "_Собрались здесь все для общего обряда\\._\n"
        "_Благословеньем свыше наделена_\n"
        "_В ней воля рока, случая игра\\._\n"
        "_Хоть \"кость\" зовут, не не скелета часть,_\n"
        "_Удача \\- твой короткий путь, сбрось свой ответ, отдавши власть\\._\""
    )
    await update.message.reply_text(text1, parse_mode=ParseMode.MARKDOWN_V2)
    await update.message.reply_text(text2, parse_mode=ParseMode.MARKDOWN_V2)
    await update.message.reply_text(poem, parse_mode=ParseMode.MARKDOWN_V2)
    return QUESTION_1

# Question 1 handler
async def handle_question_1(update: Update, context: ContextTypes.DEFAULT_TYPE) -> int:
    user_message = update.message.text
    dice_value = update.message.dice.value if update.message.dice else None

    # Case 1: Correct answer "Игровая кость"
    if user_message and user_message.lower() == "игровая кость":
        await update.message.reply_text("Так сбросьте ее.")
        return QUESTION_1

    # Case 2: Dice emoji with value other than 6
    elif dice_value is not None and dice_value != 6:
        await asyncio.sleep(3)
        await update.message.reply_text("Сегодня удача не на Вашей стороне, хах.")
        return QUESTION_1

    # Case 3: Dice emoji with value 6 - proceed to next question
    elif dice_value == 6:
        await asyncio.sleep(3)
        # Send question 2
        question_2_text = (
            "Попробуйте решить две мои загадки.\n\n"
            "1. В новой оружейной, которую я построил для своего драгоценного мужа, гэгэ насчитал всего около 93811 мечей, разного типа оружия и давно забытых бесценных артефактов. \n"
            "Если я собрал всю эту коллекцию за 6 лет и 310 дней, скольско в среднем артефактов я покупал в день?\n\n"
            "2. Однажды, встретились случайно на ярмарке Чу Ваньнин и Шэнь Циньцю. В итоге вышло так, что скупили они 63482 сладостей вместе (Шэнь Циньцю своим ученикам, а Чу Ваньнин себе), и принялись делить все поровну. Если у господин Огурца (Шэня) 50 учеников, и каждому он хочет дарить в среднем по 5 сладостей в день, на сколько дней ему хватит этих сладостей?\n\n"
            "Пишите ответы полностью, сохраняя пунктуацию."
        )
        await update.message.reply_text(question_2_text)
        return QUESTION_2

    # Case 4: Any other answer - random incorrect response
    else:
        random_response = random.choice(INCORRECT_RESPONSES)
        await update.message.reply_text(random_response, parse_mode=ParseMode.MARKDOWN_V2)
        return QUESTION_1

# Question 2 handler
async def handle_question_2(update: Update, context: ContextTypes.DEFAULT_TYPE) -> int:
    user_message = update.message.text

    # Case 1: First correct answer
    if user_message == "37.5244, 126.9640":
        await update.message.reply_text("Правильно, но посмотрите на ответы под несколько другим углом.")
        return QUESTION_2

    # Case 2: Second correct answer - proceed to next question
    elif user_message and user_message.lower() == "hybe":
        await update.message.reply_text("Поздравляю! Вы добрались до штаб-квартиры HYBE в Сеуле. Спасибо, что выбрали Gambling Dice Airlines.")

        # Send question 3
        question_3_text = (
            "Чтобы открыть двери главного входа, Вам, к сожалению, нужен код\\. Подсказку как открыть двери Вам оставила Айко\\.\n"
            "\n\n"
        )
        question_3_hint = (
            "*_Подсказка: не забудьте, где вы находитесь, и на каком языке говорят незнакомцы\\._*"
        )
        await update.message.reply_text(question_3_text, parse_mode=ParseMode.MARKDOWN_V2)
        await update.message.reply_text(question_3_hint, parse_mode=ParseMode.MARKDOWN_V2)

        # Send photo (replace 'question3_photo.jpg' with your actual photo file)
        try:
            with open('question3_photo.jpg', 'rb') as photo:
                await update.message.reply_photo(photo)
        except FileNotFoundError:
            await update.message.reply_text("【Фото с подсказкой】")

        return QUESTION_3

    # Case 3: Any other answer - random incorrect response
    else:
        random_response = random.choice(INCORRECT_RESPONSES)
        await update.message.reply_text(random_response, parse_mode=ParseMode.MARKDOWN_V2)
        return QUESTION_2

# Question 3 handler
async def handle_question_3(update: Update, context: ContextTypes.DEFAULT_TYPE) -> int:
    user_message = update.message.text

    # Case 1: Correct answer - proceed to next question
    if user_message == "1172":
        question_4_text1 = (
            "О, так Вы смогли открыть дверь\\. Я приятно удивлен\\. Хм, что же тут за песня играет в здании?\n\n"
        )
        question_4_text2 = (
            "ㅡㅡ ㅇㅡ ㅡㅡㅇ ㅇㅇ ㅡㅇㅡㅇ / ㅇㅇㅇ ㅇㅇㅇㅇ ㅡㅡㅡ ㅇㅡㅡㅇ\n\n"
        )
        question_4_hint = (
            "*_Подсказка: помните о важности такта и ритма\\._*"
        )
        await update.message.reply_text(question_4_text1, parse_mode=ParseMode.MARKDOWN_V2)
        await update.message.reply_text(question_4_text2, parse_mode=ParseMode.MARKDOWN_V2)
        await update.message.reply_text(question_4_hint, parse_mode=ParseMode.MARKDOWN_V2)
        return QUESTION_4

    # Case 2: Any other answer - random incorrect response
    else:
        random_response = random.choice(INCORRECT_RESPONSES)
        await update.message.reply_text(random_response, parse_mode=ParseMode.MARKDOWN_V2)
        return QUESTION_3

# Question 4 handler
async def handle_question_4(update: Update, context: ContextTypes.DEFAULT_TYPE) -> int:
    user_message = update.message.text

    # Case 1: Correct answer - proceed to next question
    if user_message and user_message.lower() ==  "magic shop":
        question_5_text = (
            "_Стратег, чьи планы \\- тонкая игра обмана_\n"
            "_Темный плащ оставил позади, как и в сердце рану\\._\n"
            "_Лишь не знает он, что не спастись от вечной тени_\n"
            "_Двойного черного на себе, не укрыться в свете\\._\n"
            "_Ищет смерти \\- срыв, но спасает от беды,_\n"
            "_Гений под повязкой, скрыл свои следы\\._\n"
            "_Небытье в его руках, способностей отмена,_\n"
            "_Кто ж он, циник с сердцем? Отгадай мгновенно\\._"
        )
        await update.message.reply_text(question_5_text, parse_mode=ParseMode.MARKDOWN_V2)
        return QUESTION_5

    # Case 2: Any other answer - random incorrect response
    else:
        random_response = random.choice(INCORRECT_RESPONSES)
        await update.message.reply_text(random_response, parse_mode=ParseMode.MARKDOWN_V2)
        return QUESTION_4

# Question 5 handler
async def handle_question_5(update: Update, context: ContextTypes.DEFAULT_TYPE) -> int:
    user_message = update.message.text

    # Case 1: Correct answer - proceed to question 6
    if user_message and user_message.lower() ==  "дазай осаму":
        question_6_text1 = (
            "Не думал, что Вы доберетесь до Японии сами, но Вы смогли\\. Какой сюрприз\\. Мы находимся в баре *\"Люпин\"*, в который почему\\-то часто захаживает Дазай\\. Он то и дал Айко это задание,  вместе с Анго, сказав, что что\\-то среди этого может помочь найти того, кто всю эту заварушку и начал\\. Айко почему\\-то решила оставить документы задания здесь, видимо она их и получила\\. Однако, чтобы получить эти документы, а они важны для задания, вам стоит открыть сейф при помощи кода, увы\\. Он довольно легкий\\. Айко оставила подсказку как найти код, смотрите:\n\n"
            "*1?2 12? 35? 249 5?63*\n\n"
        )
        question_6_text2 = (
            "А если даже при помощи этой очевидной подсказки не можете решить код, то Айко просила передать:\n"
            "*_\"С какой цифрой ассоциируются BTS?\"_*"
        )
        await update.message.reply_text(question_6_text1, parse_mode=ParseMode.MARKDOWN_V2)
        await update.message.reply_text(question_6_text2, parse_mode=ParseMode.MARKDOWN_V2)
        return QUESTION_6

    # Case 2: Any other answer - random incorrect response
    else:
        random_response = random.choice(INCORRECT_RESPONSES)
        await update.message.reply_text(random_response, parse_mode=ParseMode.MARKDOWN_V2)
        return QUESTION_5

# Question 6 handler
async def handle_question_6(update: Update, context: ContextTypes.DEFAULT_TYPE) -> int:
    user_message = update.message.text

    # Case 1: Correct answer - proceed to question 7
    if user_message == "4846":
        question_7_part1 = (
            "Хм\\.\\.\\. Давайте\\-ка прочитаем документы\\. Здесь о каком\\-то сумасшедшем, что повелевает мертвыми своей флейтой, о каких\\-то друзьях? Обрезанных рукавах, которые расстались прямо перед \"KFC\", о каком\\-то духе лиса, что ищет свое божество\\.\\.\\. Все не то\\.\n"
        )
        await update.message.reply_text(question_7_part1, parse_mode=ParseMode.MARKDOWN_V2)

        await update.message.reply_text("О, нашел\\. Написано, что нужно освободить какого\\-то человека из тюрьмы\\. Имя неизвестно, происхождение неизвестно, назвался *_Никем_*, и ищет путь домой\\.", parse_mode=ParseMode.MARKDOWN_V2)

        question_7_part2 = (
            "Ну вот и камера тюрьмы, откуда Вам стоит вызволить этого потерянного бедолагу. Попробуйте открыть камеру, используя код на картинке."
        )
        await update.message.reply_text(question_7_part2)

        # Send photo for question 7 (replace 'question7_photo.jpg' with your actual photo file)
        try:
            with open('question7_photo.jpg', 'rb') as photo:
                await update.message.reply_photo(photo)
        except FileNotFoundError:
            await update.message.reply_text("【Фото с кодом для тюремной камеры】")

        return QUESTION_7

    # Case 2: Any other answer - random incorrect response
    else:
        random_response = random.choice(INCORRECT_RESPONSES)
        await update.message.reply_text(random_response, parse_mode=ParseMode.MARKDOWN_V2)
        return QUESTION_6

# Question 7 handler
async def handle_question_7(update: Update, context: ContextTypes.DEFAULT_TYPE) -> int:
    user_message = update.message.text

    # Case 1: Correct answer - proceed to question 8
    if user_message == "8721":
        question_8_text = (
            "Поздравляю, вы открыли камеру от тюрьмы\\. Но какой\\-то он потрепанный, словно *_двадцать лет_* не видел зеркала и не держал в руках расческу\\. Как думаете, кто он?\n\n"
        )
        question_8_hint = (
            "*_Подсказка от Айко: внимательно читайте реплики Сань Лана, и можете догадаться, о ком идет речь\\._*"
        )
        await update.message.reply_text(question_8_text, parse_mode=ParseMode.MARKDOWN_V2)
        await update.message.reply_text(question_8_hint, parse_mode=ParseMode.MARKDOWN_V2)
        return QUESTION_8

    # Case 2: Any other answer - random incorrect response
    else:
        random_response = random.choice(INCORRECT_RESPONSES)
        await update.message.reply_text(random_response, parse_mode=ParseMode.MARKDOWN_V2)
        return QUESTION_7

# Question 8 handler
async def handle_question_8(update: Update, context: ContextTypes.DEFAULT_TYPE) -> int:
    user_message = update.message.text

    # Case 1: Correct answer - proceed to question 9
    if user_message and user_message.lower() == "одиссей":
        question_9_part1 = (
            "Так уж и быть, позволю в последний раз путешествовать через мой Gambling Dice Airlines. Пункт назначения - Итака, вернем Одиссея в его королевство."
        )
        await update.message.reply_text(question_9_part1)

        # Send audio file (replace 'question9_audio.mp3' with your actual audio file)
        try:
            with open('Full Speed Ahead.mp3', 'rb') as audio:
                await update.message.reply_audio(audio)
        except FileNotFoundError:
            await update.message.reply_text("【Аудио файл】")

        question_9_part2 = (
            "Какая незадача, Одиссей каким\\-то образом успел прогневать Посейдона, из\\-за чего начался *_Шторм_*\\. И что за напасть, в самый неподходящий момент Ваша *_Удача Закончилась_* \\(делиться ей я не намерен, это только для гэгэ\\)\\. Теперь Вам предстоит пройти через *_Игру Богов_*, чтобы не оказаться в воде, и вернуть Одиссея домой к Пенелопе\\."
        )
        await update.message.reply_text(question_9_part2, parse_mode=ParseMode.MARKDOWN_V2)

        await update.message.reply_text("??????")

        question_9_part3 = "*_Подсказка: некоторые слова имеют большую силу\\._*"
        await update.message.reply_text(question_9_part3, parse_mode=ParseMode.MARKDOWN_V2)

        return QUESTION_9

    # Case 2: Any other answer - random incorrect response
    else:
        random_response = random.choice(INCORRECT_RESPONSES)
        await update.message.reply_text(random_response, parse_mode=ParseMode.MARKDOWN_V2)
        return QUESTION_8

# Question 9 handler
async def handle_question_9(update: Update, context: ContextTypes.DEFAULT_TYPE) -> int:
    user_message = update.message.text

    # Case 1: Correct answer - proceed to question 10 (final question)
    if user_message == "101130":
        question_10_text = (
            "Вы успешно вернули Одиссея в Итаку, поздравляю\\! А озорником, который решил затеять весь этот квест, оказался Гермес, которому стало скучно\\. По словам Гермеса, теперь, для того, чтобы вернуть все в норму, нужно написать Айко *_\"С днем рождения\\!\"_* на всех языках, которые она знает или учит\\. Поторопитесь\\! Победителем станет тот, кто окажется быстрее всех\\!"
        )
        await update.message.reply_text(question_10_text, parse_mode=ParseMode.MARKDOWN_V2)
        return QUESTION_10

    # Case 2: Any other answer - random incorrect response
    else:
        random_response = random.choice(INCORRECT_RESPONSES)
        await update.message.reply_text(random_response, parse_mode=ParseMode.MARKDOWN_V2)
        return QUESTION_9

# Question 10 handler - This is the final question where any message could be considered as participation
async def handle_question_10(update: Update, context: ContextTypes.DEFAULT_TYPE) -> int:
    user_message = update.message.text

    # For question 10, we accept any message as participation in the final challenge
    # You might want to add specific logic here if needed
    await update.message.reply_text("Спасибо за участие в квесте!")
    return ConversationHandler.END

# Cancel command
async def cancel(update: Update, context: ContextTypes.DEFAULT_TYPE) -> int:
    """Cancels and ends the conversation."""
    user = update.message.from_user
    logger.info("User %s canceled the conversation.", user.first_name)
    await update.message.reply_text("До свидания! Если захотите попробовать снова, напишите /start")
    return ConversationHandler.END

def main() -> None:
    """Run the bot."""
    # Create the Application and pass it your bot's token.
    application = Application.builder().token("8294505412:AAF13c_ECMJUC5lNgLc2b1lhNtSH5CQ6xME").build()

    # Add conversation handler with the states
    conv_handler = ConversationHandler(
        entry_points=[CommandHandler("start", start)],
        states={
            QUESTION_1: [
                MessageHandler(DICE_FILTER, handle_question_1),
                MessageHandler(filters.TEXT & ~filters.COMMAND, handle_question_1)
            ],
            QUESTION_2: [
                MessageHandler(filters.TEXT & ~filters.COMMAND, handle_question_2)
            ],
            QUESTION_3: [
                MessageHandler(filters.TEXT & ~filters.COMMAND, handle_question_3)
            ],
            QUESTION_4: [
                MessageHandler(filters.TEXT & ~filters.COMMAND, handle_question_4)
            ],
            QUESTION_5: [
                MessageHandler(filters.TEXT & ~filters.COMMAND, handle_question_5)
            ],
            QUESTION_6: [
                MessageHandler(filters.TEXT & ~filters.COMMAND, handle_question_6)
            ],
            QUESTION_7: [
                MessageHandler(filters.TEXT & ~filters.COMMAND, handle_question_7)
            ],
            QUESTION_8: [
                MessageHandler(filters.TEXT & ~filters.COMMAND, handle_question_8)
            ],
            QUESTION_9: [
                MessageHandler(filters.TEXT & ~filters.COMMAND, handle_question_9)
            ],
            QUESTION_10: [
                MessageHandler(filters.TEXT & ~filters.COMMAND, handle_question_10)
            ],
        },
        fallbacks=[CommandHandler("cancel", cancel)],
    )

    application.add_handler(conv_handler)

    application.run_polling(allowed_updates=Update.ALL_TYPES, close_loop=False)

    # Run the bot until the user presses Ctrl-C
    application.run_polling(allowed_updates=Update.ALL_TYPES)

if __name__ == "__main__":
    # Check if we're in an environment with an existing event loop
    try:
        import nest_asyncio
        nest_asyncio.apply()
        print("Applied nest_asyncio for existing event loop")
    except ImportError:
        print("nest_asyncio not available, running without it")

    main()

<>:46: SyntaxWarning: invalid escape sequence '\.'
<>:46: SyntaxWarning: invalid escape sequence '\.'
/tmp/ipython-input-1839044636.py:46: SyntaxWarning: invalid escape sequence '\.'
  "Добро пожаловать в Призрачный Город\. По просьбе Айко я буду помогать вам с вашим квестом\. Вам нужно будет решить пару загадок, и ответить на них правильно, ничего сложного, даже шиди моего подопечного справился бы с этим спустя рукава\.\n"


Applied nest_asyncio for existing event loop


In [1]:
pip install python-telegram-bot

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 731.0/731.0 kB 17.0 MB/s eta 0:00:00


In [4]:
pip show python-telegram-bot

Name: python-telegram-bot
Version: 22.5
Summary: We have made you a wrapper you can't refuse
Home-page: https://python-telegram-bot.org
Author: 
Author-email: Leandro Toledo <devs@python-telegram-bot.org>
License: 
Location: /usr/local/lib/python3.12/dist-packages
Requires: httpx
Required-by: 


In [5]:
pip install --upgrade python-telegram-bot